# MODELING EXPERIMENTS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

from fake_news_co.paths import DATA_PROCESSED # Export the path to the processed data directory

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "dataset.csv")

le = LabelEncoder()

df["label_id"] = le.fit_transform(df["label"])   # 0,1,2 alfabético
# le.classes_ guarda el mapeo para la matriz de confusión y la demo
train = df[df["split"] == "train"][["text", "label_id"]]
val = df[df["split"] == "val"][["text", "label_id"]]
test = df[df["split"] == "test"][["text", "label_id"]]

print(f"Train shape: {train.shape}")
print(f"Validation shape: {val.shape}")
print(f"Test shape: {test.shape}")

X_train, y_train = train["text"], train["label_id"]
X_val, y_val = val["text"], val["label_id"]
X_test, y_test = test["text"], test["label_id"]

# 1. TF-IDF BASELINE

## 1.1 TF-IDF 

In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline


pipe_tfid = Pipeline(
    [("tfidf", TfidfVectorizer(min_df=2, ngram_range=(1, 2))), 
     ("clf", LogisticRegression(max_iter=100_000, random_state=42, class_weight="balanced", solver="lbfgs"))]
)

params = {
    "clf__C": [0.01, 0.1, 1, 10]
    }

grid_tfidf = GridSearchCV(
    pipe_tfid, 
    params, cv=3, scoring="f1_macro", n_jobs=-1, verbose=1
)
time_start = time.time()
grid_tfidf.fit(X_train, y_train)
time_end = time.time()
print(f"Grid search time: {time_end - time_start} seconds")

best_params = grid_tfidf.best_params_
print(f"Best parameters: {best_params}")

best_estimator = grid_tfidf.best_estimator_

In [ ]:
# F1-Macro on validation set

y_val_pred = best_estimator.predict(X_val)
val_f1 = f1_score(y_val, y_val_pred, average="macro")   # (verdad, predicción)
print(f"Macro-F1 on validation: {val_f1:.4f}")

from sklearn.metrics import classification_report
print(classification_report(y_val, y_val_pred, digits=3, target_names=le.classes_))

## 1.2 TF-IDF SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

pipe_smote = ImbPipeline([
    ("tfidf", TfidfVectorizer(min_df=2, ngram_range=(1, 2))),
    ("smote", SMOTE(random_state=42)),
    ("clf",   LogisticRegression(max_iter=100_000, random_state=42)),
])

params = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__solver": ["lbfgs", "saga"],
}

grid_smote = GridSearchCV(
    pipe_smote,
    params, cv=3, scoring="f1_macro", n_jobs=-1, verbose=1
)
time_start = time.time()
grid_smote.fit(X_train, y_train)
time_end = time.time()
print(f"Grid search time: {time_end - time_start} seconds")

best_params = grid_smote.best_params_
print(f"Best parameters: {best_params}")

best_estimator_smote = grid_smote.best_estimator_

In [ ]:
# F1-Macro on validation set

y_val_pred = best_estimator_smote.predict(X_val)
val_f1_smote = f1_score(y_val, y_val_pred, average="macro")   # (verdad, predicción)
print(f"Macro-F1 on validation: {val_f1_smote:.4f}")

print(classification_report(y_val, y_val_pred, digits=3, target_names=le.classes_))

## 1.3 COMPARISON TF-IDF

In [ ]:
# Comparison

f1_tfidf = pd.DataFrame({
    "F1-Macro Val": [val_f1, val_f1_smote],
    "Model Name": ["LR (TF-IDF)", "LR (TF-IDF + SMOTE)"],
}).set_index("Model Name").sort_values("F1-Macro Val", ascending=False)

print("\nComparison of F1-Macro scores on validation set:")
print(f1_tfidf.to_markdown())

plt.figure(figsize=(8, 6))
sns.barplot(data=f1_tfidf, x="Model Name", y="F1-Macro Val", color="salmon")
bars = plt.gca().patches
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, round(yval, 4), ha="center", va="bottom", fontsize=10, fontweight="bold")
sns.despine()
plt.title("Comparison of F1-Macro scores on validation set", fontsize=16, fontweight="bold", pad=20)
plt.xlabel("")
plt.show()


- Introducing SMOTE technique did not improve significantly F1-Macro metric, for this reason no SMOTE (synthetic data) technique is selected.

# 2. BETO

## 